In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from matplotlib import colors
from matplotlib.colors import LinearSegmentedColormap
from tqdm.auto import tqdm

from shaft_force_sensing.data import get_train_test, get_cols, SensorDataset

In [ ]:
ROOT = Path('..')
DATA_ROOT = ROOT / 'data'

In [ ]:
train_paths, test_paths = get_train_test(DATA_ROOT, '')

In [ ]:
groups = [
    'Free',
    'Palpation',
    'Traction'
]
groups = dict(zip(groups, [0] * len(groups)))

In [ ]:
for p in tqdm(train_paths):
    g = re.match(r'([a-zA-Z]+)_\d+', p.stem).groups()[0]
    df = pd.read_csv(p)
    groups[g] += len(df)

Train

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    length = v * 0.9
    if k == 'Free':
        length /= 4
    subset_lengths[k] = length

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Val

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    length = v * 0.1
    if k == 'Free':
        length /= 4
    subset_lengths[k] = length

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Test

In [ ]:
groups = [
    'Free',
    'Palpation',
    'Traction'
]
groups = dict(zip(groups, [0] * len(groups)))

In [ ]:
for p in tqdm(test_paths):
    g = re.match(r'([a-zA-Z]+)_\d+', p.stem).groups()[0]
    df = pd.read_csv(p)
    groups[g] += len(df)

In [ ]:
subset_lengths = {}
for k, v in groups.items():
    subset_lengths[k] = v

total_length = sum(subset_lengths.values())
for k, length in subset_lengths.items():
    ratio = (length / total_length) * 100 if total_length else 0.0
    print(
        f"Group: {k}, Length: {length/100/60:.1f} min, "
        f"Ratio: {ratio:.1f}%"
    )
print(f"Total Length: {total_length/100/60:.1f} min")

Distribution of the dataset

In [ ]:
hex10_forces = np.zeros((0, 3))
ati_forces = np.zeros((0, 3))

for p in tqdm(test_paths):
    df = pd.read_csv(p)
    hex10_forces = np.vstack((hex10_forces, df[['fx', 'fy', 'fz']].values))
    ati_forces = np.vstack((ati_forces, df[['ati_fx', 'ati_fy', 'ati_fz']].values))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase
from matplotlib.ticker import LogLocator, LogFormatterSciNotation, NullFormatter

# Single-column friendly size for double-column papers
fig, axes = plt.subplots(3, 1, figsize=(5, 3), sharex=True, dpi=300)
axis_names = ['x', 'y', 'z']
component_colors = {'x': '#b23a2a', 'y': '#2f7a53', 'z': '#2f3fa8'}
colors_map = {'ATI': '#5c5c5c'}
force_pairs = [
    ('ATI', ati_forces),
    ('HEX10', hex10_forces),
]

all_forces = np.concatenate([ati_forces.reshape(-1), hex10_forces.reshape(-1)])
if all_forces.size > 0:
    x_min = np.floor(all_forces.min() / 10.0) * 10.0
    x_max = np.ceil(all_forces.max() / 10.0) * 10.0
    if x_min == x_max:
        x_min -= 10.0
        x_max += 10.0
else:
    x_min, x_max = -10.0, 10.0

# Keep x ticks readable while forcing both extremes to appear.
span = x_max - x_min
tick_step = max(10.0, np.ceil(span / 50.0) * 10.0)
x_ticks = np.arange(x_min, x_max + 0.1, tick_step)
if x_ticks.size == 0 or x_ticks[-1] < x_max:
    x_ticks = np.append(x_ticks, x_max)
x_ticks = np.unique(x_ticks)

for axis_idx, axis_name in enumerate(axis_names):
    ax = axes[axis_idx]
    for label, force_values in force_pairs:
        color = component_colors[axis_name] if label == 'HEX10' else colors_map[label]
        ax.hist(
            force_values[:, axis_idx],
            bins=70,
            alpha=0.7 if label == 'HEX10' else 1.0,
            color=color,
            edgecolor='none',
            zorder=3 if label == 'HEX10' else 2,
        )
    ax.set_xlim(x_min, x_max + 0.5)
    ax.set_xticks(x_ticks)
    ax.text(
        0.04,
        0.60,
        f'$F_{axis_name}$',
        transform=ax.transAxes,
        fontsize=12,
        fontweight='bold',
        color=component_colors[axis_name],
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=2.5),
    )
    ax.set_yscale('log')
    ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0,), numticks=4))
    ax.yaxis.set_major_formatter(LogFormatterSciNotation(base=10, labelOnlyBase=True))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1, numticks=8))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, which='major', linestyle='--', linewidth=0.8, color='#c4c4c4', alpha=0.45)
    ax.grid(False, which='minor')
    ax.tick_params(axis='both', labelsize=12, width=1.1, length=4, color='black')
    for spine in ax.spines.values():
        spine.set_color('black')
        spine.set_linewidth(1.1)

class HandlerTriColor(HandlerBase):
    def create_artists(
        self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans
    ):
        stripe_width = width / 3.0
        artists = []
        for i, c in enumerate(orig_handle):
            artists.append(
                Rectangle(
                    (xdescent + i * stripe_width, ydescent),
                    stripe_width,
                    height,
                    facecolor=c,
                    edgecolor='none',
                    transform=trans,
                )
            )
        return artists

axes[-1].set_xlabel('Force (N)', fontsize=12)
fig.supylabel('Frequency', fontsize=12, x=0.02)

legend_handles = [
    Patch(facecolor=colors_map['ATI'], edgecolor='none'),
    (component_colors['x'], component_colors['y'], component_colors['z']),
]
legend_labels = ['ATI', 'HEX10']

fig.legend(
    legend_handles,
    legend_labels,
    loc='upper center',
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
    fontsize=12,
    handler_map={tuple: HandlerTriColor()},
)

plt.tight_layout(rect=[0.0, 0.0, 1.0, 0.95])
output_path = Path('../logs/results/hex10_vs_ati_hist.pdf')
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, format='pdf', bbox_inches='tight')
plt.show()